# Laboratory Work #2 — Fix Encoding & Clean Text with Regex

## Assignment

**Topic:** Repairing broken text encoding and cleaning noisy text

**Goal:** Learn how to restore readable text from a wrong encoding and remove garbage with regular expressions.

### Tasks
1. Take a short text fragment saved in the **wrong encoding** (mojibake / "krakozyabry").
2. Convert it to the **correct encoding**.
3. Clean the text from noise using **regular expressions** (HTML tags, links, repeated symbols, extra spaces, etc.).
4. Write short conclusions about what was wrong and what became readable.

> This notebook is an **example solution**. Comments and explanations are in English.


## 0. Import libraries


In [1]:
# re = regular expressions for text cleaning
import re

print("Library imported successfully.")


Library imported successfully.


## 1. Broken text (wrong encoding)

Below is an example of a product review that was originally written in **UTF-8**,
but opened as **Latin-1 (ISO-8859-1)**. Because of that, Cyrillic letters look like garbage.

The text also contains noise:
- HTML tags
- a spam link
- repeated punctuation
- extra spaces


In [2]:
# Garbled text: UTF-8 bytes were incorrectly decoded as Latin-1.
# NOTE: we generate the mojibake in code instead of pasting it,
# because it contains invisible control characters (e.g. \x85)
# that Colab treats as line breaks -> "SyntaxError: unterminated string literal".
original_review = "Отзыв о наушниках!!!   <br><b>Очень доволен</b>  звуком и микрофоном......   http://spam-deals.example.com/now  ##### Батарея держит 2 дня!!!   email_me@@@test!!!   Рейтинг::::5"

# Simulate the mistake: write as UTF-8, read as Latin-1
broken_text = original_review.encode("utf-8").decode("latin-1")

print("Broken text:")
print(broken_text)


Broken text:
ÐÑÐ·ÑÐ² Ð¾ Ð½Ð°ÑÑÐ½Ð¸ÐºÐ°Ñ!!!   <br><b>ÐÑÐµÐ½Ñ Ð´Ð¾Ð²Ð¾Ð»ÐµÐ½</b>  Ð·Ð²ÑÐºÐ¾Ð¼ Ð¸ Ð¼Ð¸ÐºÑÐ¾ÑÐ¾Ð½Ð¾Ð¼......   http://spam-deals.example.com/now  ##### ÐÐ°ÑÐ°ÑÐµÑ Ð´ÐµÑÐ¶Ð¸Ñ 2 Ð´Ð½Ñ!!!   email_me@@@test!!!   Ð ÐµÐ¹ÑÐ¸Ð½Ð³::::5


## 2. Fix the encoding

Idea:
1. Encode the broken string back to bytes using `latin-1`
2. Decode those bytes as `utf-8`

This restores the original Cyrillic text.


In [3]:
# Step A: turn mojibake characters back into the original bytes
raw_bytes = broken_text.encode("latin-1")

# Step B: decode bytes with the correct encoding (UTF-8)
fixed_text = raw_bytes.decode("utf-8")

print("Text after encoding fix:")
print(fixed_text)


Text after encoding fix:
Отзыв о наушниках!!!   <br><b>Очень доволен</b>  звуком и микрофоном......   http://spam-deals.example.com/now  ##### Батарея держит 2 дня!!!   email_me@@@test!!!   Рейтинг::::5


## 3. Clean garbage with regular expressions

Now the letters are readable, but the text is still noisy.
We remove:
1. HTML tags
2. URLs
3. email-like / spam tokens
4. repeated `#` and `:` symbols
5. too many repeated punctuation marks
6. extra spaces


In [4]:
# Work with a copy so we can compare before / after
clean_text = fixed_text

# 1) Remove HTML tags like <br>, <b>, </b>
clean_text = re.sub(r"<[^>]+>", " ", clean_text)

# 2) Remove URLs starting with http:// or https://
clean_text = re.sub(r"https?://\S+", " ", clean_text)

# 3) Remove simple spam tokens with @@@ or email-like junk
clean_text = re.sub(r"\S*@{2,}\S*", " ", clean_text)

# 4) Remove long runs of # or :
clean_text = re.sub(r"[#:]{2,}", " ", clean_text)

# 5) Replace 3+ repeated punctuation marks with a single one
# Example: !!!!!! -> !    ...... -> .
clean_text = re.sub(r"([!?.,])\1{2,}", r"\1", clean_text)

# 6) Collapse multiple spaces into one space and trim edges
clean_text = re.sub(r"\s+", " ", clean_text).strip()

print("Clean text:")
print(clean_text)


Clean text:
Отзыв о наушниках! Очень доволен звуком и микрофоном. Батарея держит 2 дня! Рейтинг 5


## 4. Compare results


In [5]:
print("=== BEFORE (broken encoding) ===")
print(broken_text)
print()

print("=== AFTER encoding fix ===")
print(fixed_text)
print()

print("=== AFTER regex cleaning ===")
print(clean_text)
print()

print("Characters before cleaning:", len(fixed_text))
print("Characters after cleaning:", len(clean_text))


=== BEFORE (broken encoding) ===
ÐÑÐ·ÑÐ² Ð¾ Ð½Ð°ÑÑÐ½Ð¸ÐºÐ°Ñ!!!   <br><b>ÐÑÐµÐ½Ñ Ð´Ð¾Ð²Ð¾Ð»ÐµÐ½</b>  Ð·Ð²ÑÐºÐ¾Ð¼ Ð¸ Ð¼Ð¸ÐºÑÐ¾ÑÐ¾Ð½Ð¾Ð¼......   http://spam-deals.example.com/now  ##### ÐÐ°ÑÐ°ÑÐµÑ Ð´ÐµÑÐ¶Ð¸Ñ 2 Ð´Ð½Ñ!!!   email_me@@@test!!!   Ð ÐµÐ¹ÑÐ¸Ð½Ð³::::5

=== AFTER encoding fix ===
Отзыв о наушниках!!!   <br><b>Очень доволен</b>  звуком и микрофоном......   http://spam-deals.example.com/now  ##### Батарея держит 2 дня!!!   email_me@@@test!!!   Рейтинг::::5

=== AFTER regex cleaning ===
Отзыв о наушниках! Очень доволен звуком и микрофоном. Батарея держит 2 дня! Рейтинг 5

Characters before cleaning: 177
Characters after cleaning: 85


## 5. Conclusions

1. The original fragment looked unreadable because UTF-8 text was opened as Latin-1.
2. The fix `encode("latin-1").decode("utf-8")` restored normal Cyrillic letters.
3. Regular expressions removed HTML, links, spam tokens, and repeated symbols.
4. After cleaning, the review is short, readable, and ready for further analysis (for example, sentiment or rating extraction).
5. Encoding problems and text noise often appear together in real scraped data, so both steps are useful in practice.


---

## Independent assignment for students

<!--
STUDENT TASK (independent work)

Reproduce the same laboratory work for ANOTHER broken text.

Important difference:
- In the example above, the mistake was: UTF-8 read as Latin-1.
- In your task, the mistake is different: Windows-1251 (cp1251) text was read as Latin-1.

What you need to do:
1. Take the broken text from the next code cell.
2. Restore the correct encoding.
   Hint for this task:
       text.encode("latin-1").decode("cp1251")
3. Clean the restored text with regular expressions
   (HTML tags, URLs, repeated symbols, extra spaces, etc.).
4. Print before / after results.
5. Write 5–7 conclusions in English about:
   - what encoding error you fixed
   - what garbage you removed
   - whether the final text is good enough for analysis

Submit: Colab notebook with code + conclusions.
-->

**Independent task:** fix and clean **another** text with a **different** encoding error.

In the example, UTF-8 was read as Latin-1.
In your task, **Windows-1251 (cp1251)** was read as Latin-1.

Use the text below, restore encoding, clean it with regex, and evaluate the quality of the final result.


In [6]:
# ============================================================
# TEXT FOR INDEPENDENT STUDENT WORK
# Error type: cp1251 text incorrectly decoded as latin-1
# Students should NOT receive the already fixed version.
# ============================================================

student_broken_text = "Ðåöåíçèÿ íà ôèëüì!!! <p>Ñþæåò ñèëüíûé</p>, àêòåðñêàÿ èãðà õîðîøàÿ......   Ñì. òàêæå https://fake-movies.example.org/top ***** Íî êîíåö ñëèøêîì çàòÿíóò!!!!  contact###admin@@@mail  Îöåíêà::::8/10"

print("Broken text for independent work:")
print(student_broken_text)

# Hint (students should write the solution themselves):
# fixed = student_broken_text.encode("latin-1").decode("cp1251")
# then clean with re.sub(...)


Broken text for independent work:
Ðåöåíçèÿ íà ôèëüì!!! <p>Ñþæåò ñèëüíûé</p>, àêòåðñêàÿ èãðà õîðîøàÿ......   Ñì. òàêæå https://fake-movies.example.org/top ***** Íî êîíåö ñëèøêîì çàòÿíóò!!!!  contact###admin@@@mail  Îöåíêà::::8/10


### Expected student steps (short checklist)

1. Print the broken text.
2. Fix encoding with `encode("latin-1").decode("cp1251")`.
3. Remove HTML / links / repeated symbols / extra spaces with `re.sub`.
4. Compare lengths and readability before vs after.
5. Write conclusions in English.

**End of Laboratory Work #2 example solution.**


---

# My Solution — Independent Task

**Error type:** Windows-1251 (cp1251) text was decoded as Latin-1.


## Step 1. Print the broken text


In [7]:
print("Broken text:")
print(student_broken_text)
print("Length:", len(student_broken_text))


Broken text:
Ðåöåíçèÿ íà ôèëüì!!! <p>Ñþæåò ñèëüíûé</p>, àêòåðñêàÿ èãðà õîðîøàÿ......   Ñì. òàêæå https://fake-movies.example.org/top ***** Íî êîíåö ñëèøêîì çàòÿíóò!!!!  contact###admin@@@mail  Îöåíêà::::8/10
Length: 194


## Step 2. Fix the encoding

In cp1251, **one Cyrillic letter = one byte**. Latin-1 also maps one byte to one character,
so each Cyrillic letter became one Latin letter (`Ð`, `å`, `ö`...).

Fix:
1. `encode("latin-1")` → get the original bytes back
2. `decode("cp1251")` → read those bytes with the correct encoding


In [8]:
# Step A: mojibake -> original bytes
student_bytes = student_broken_text.encode("latin-1")

# Step B: bytes -> correct text (Windows-1251)
student_fixed = student_bytes.decode("cp1251")

print("Text after encoding fix:")
print(student_fixed)


Text after encoding fix:
Рецензия на фильм!!! <p>Сюжет сильный</p>, актерская игра хорошая......   См. также https://fake-movies.example.org/top ***** Но конец слишком затянут!!!!  contact###admin@@@mail  Оценка::::8/10


### Why not UTF-8 here?
If we try the method from the example (`decode("utf-8")`), it fails, because these bytes are not valid UTF-8.


In [9]:
try:
    student_bytes.decode("utf-8")
except UnicodeDecodeError as e:
    print("UTF-8 does not work:", e)


UTF-8 does not work: 'utf-8' codec can't decode byte 0xd0 in position 0: invalid continuation byte


## Step 3. Clean the text with regular expressions

We remove:
1. HTML tags (`<p>`, `</p>`)
2. The URL and the phrase "См. также" ("see also") that only points to the spam link
3. The spam token `contact###admin@@@mail`
4. Runs of `*****` and `###`
5. `::::` → `: `
6. Repeated punctuation (`!!!!` → `!`, `......` → `.`)
7. Spaces before punctuation and extra spaces


In [10]:
student_clean = student_fixed

# 1) Remove HTML tags
student_clean = re.sub(r"<[^>]+>", " ", student_clean)

# 2) Remove URLs, and the "См. также" (see also) phrase that introduces the link
student_clean = re.sub(r"https?://\S+", " ", student_clean)
student_clean = re.sub(r"См\.\s*также", " ", student_clean)

# 3) Remove spam tokens containing @@ (e.g. contact###admin@@@mail)
student_clean = re.sub(r"\S*@{2,}\S*", " ", student_clean)

# 4) Remove runs of 3+ * or # symbols
student_clean = re.sub(r"[*#]{3,}", " ", student_clean)

# 5) Replace :::: with a normal colon + space
student_clean = re.sub(r":{2,}", ": ", student_clean)

# 6) Collapse 3+ repeated punctuation into one mark
student_clean = re.sub(r"([!?.,])\1{2,}", r"\1", student_clean)

# 7) Remove spaces before punctuation, collapse spaces, trim
student_clean = re.sub(r"\s+([,.!?])", r"\1", student_clean)
student_clean = re.sub(r"\s+", " ", student_clean).strip()

print("Clean text:")
print(student_clean)


Clean text:
Рецензия на фильм! Сюжет сильный, актерская игра хорошая. Но конец слишком затянут! Оценка: 8/10


## Step 4. Compare before / after


In [11]:
print("=== BEFORE (broken encoding) ===")
print(student_broken_text)
print()
print("=== AFTER encoding fix ===")
print(student_fixed)
print()
print("=== AFTER regex cleaning ===")
print(student_clean)
print()
print("Characters before cleaning:", len(student_fixed))
print("Characters after cleaning: ", len(student_clean))
print(f"Removed: {len(student_fixed) - len(student_clean)} characters "
      f"({(1 - len(student_clean)/len(student_fixed))*100:.0f}% noise)")

# Bonus: extract the rating, to show the text is ready for analysis
m = re.search(r"Оценка:\s*(\d+)/(\d+)", student_clean)
if m:
    print("Extracted rating:", m.group(1), "out of", m.group(2))


=== BEFORE (broken encoding) ===
Ðåöåíçèÿ íà ôèëüì!!! <p>Ñþæåò ñèëüíûé</p>, àêòåðñêàÿ èãðà õîðîøàÿ......   Ñì. òàêæå https://fake-movies.example.org/top ***** Íî êîíåö ñëèøêîì çàòÿíóò!!!!  contact###admin@@@mail  Îöåíêà::::8/10

=== AFTER encoding fix ===
Рецензия на фильм!!! <p>Сюжет сильный</p>, актерская игра хорошая......   См. также https://fake-movies.example.org/top ***** Но конец слишком затянут!!!!  contact###admin@@@mail  Оценка::::8/10

=== AFTER regex cleaning ===
Рецензия на фильм! Сюжет сильный, актерская игра хорошая. Но конец слишком затянут! Оценка: 8/10

Characters before cleaning: 194
Characters after cleaning:  96
Removed: 98 characters (51% noise)
Extracted rating: 8 out of 10


## Step 5. Conclusions

1. **Encoding error:** the review was written in **Windows-1251 (cp1251)** but opened as **Latin-1**, so every Cyrillic letter turned into a Latin symbol (`Ð`, `å`, `ö`...).
2. **Fix:** `encode("latin-1").decode("cp1251")` restored the readable Russian text. The UTF-8 method from the example does **not** work here: it raises `UnicodeDecodeError`, because cp1251 uses 1 byte per letter, while UTF-8 uses 2 bytes per Cyrillic letter.
3. **How to recognise the error type:** in UTF-8→Latin-1 mojibake, every letter becomes **two** symbols (`Ð°`). In cp1251→Latin-1 mojibake, every letter becomes **one** symbol (`à`). So the broken string was about the same length as the real text.
4. **Garbage removed with regex:** HTML tags (`<p>`), a spam URL with its "См. также" phrase, the spam token `contact###admin@@@mail`, runs of `*****`, `::::`, repeated `!!!!` and `......`, and extra spaces.
5. **Result:** the text went from 194 to 96 characters (about half of it was noise). The final review reads: *"Рецензия на фильм! Сюжет сильный, актерская игра хорошая. Но конец слишком затянут! Оценка: 8/10"*.
6. **Good enough for analysis?** Yes. It is readable and keeps the useful information: the opinion (positive plot and acting, negative ending) and the rating 8/10, which we extracted with one regex. It can be used for sentiment analysis or rating prediction.
7. **Limitation:** these regex rules were written for this specific text. On real scraped data (the "garbage in, garbage out" principle from the lecture), the rules need testing on many examples, because a rule like removing `@@` tokens could also delete useful content.
